In [ ]:
import sys, os

sys.path.append(os.path.dirname(os.getcwd()))

import tqdm
import matplotlib.pyplot as plt

plt.rcParams.update({"mathtext.fontset": "cm"})

import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platforms", "cpu")

import shtns
import jaxbind

import models.imex_solver as imex
from models.sph_mhd import (
    SphericalMHD,
    dynamical_solver,
    integral,
    kinetic_energy,
)

In [ ]:
eq = SphericalMHD(Ra=100, E=1e-3, radius_ratio=0.35, l_max=31, m_max=7, m_res=4, n_r=84)

shtns_cfg = shtns.sht(eq.l_max, mres=eq.m_res)  # single configuration used throughout
shtns_cfg.set_grid(nl_order=2)

shtns_cfg.wts = shtns_cfg.gauss_wts() * (2 * np.pi) / shtns_cfg.nphi
shtns_cfg.wts = np.hstack((shtns_cfg.wts, np.flip(shtns_cfg.wts))).reshape((-1, 1))
shtns_cfg.wts_1 = 1.0 / shtns_cfg.wts
shtns_cfg._l2 = shtns_cfg.l * (shtns_cfg.l + 1.0)
shtns_cfg._l2[eq.l_max + 1 :] *= 2.0
shtns_cfg._l_2 = np.zeros_like(shtns_cfg._l2)
shtns_cfg._l_2[1:] = 1.0 / shtns_cfg._l2[1:]


def sht_synth(out, args, kwargs_dump):
    nargs = len(args)
    if nargs == 1:
        out[0][()] = shtns_cfg.synth(args[0])
    else:
        q = shtns_cfg.synth(*args)
        for i in range(nargs):
            out[i][()] = q[i]


def sht_analys(out, args, kwargs_dump):
    nargs = len(args)
    if nargs == 1:
        out[0][()] = shtns_cfg.analys(args[0])
    else:
        alm = shtns_cfg.analys(*args)
        for i in range(nargs):
            out[i][()] = alm[i]


def sht_synth_transposed(out, args, kwargs_dump):
    nargs = len(args)
    if nargs == 1:
        xlm = shtns_cfg.analys(args[0] * shtns_cfg.wts_1)
        xlm[shtns_cfg.lmax + 1 :] *= 2.0
        out[0][()] = xlm
    elif nargs == 2:
        slm, tlm = shtns_cfg.analys(
            args[0] * shtns_cfg.wts_1, args[1] * shtns_cfg.wts_1
        )
        out[0][()] = slm * shtns_cfg._l2
        out[1][()] = tlm * shtns_cfg._l2
    elif nargs == 3:
        qlm, slm, tlm = shtns_cfg.analys(
            args[0] * shtns_cfg.wts_1,
            args[1] * shtns_cfg.wts_1,
            args[2] * shtns_cfg.wts_1,
        )
        qlm[shtns_cfg.lmax + 1 :] *= 2.0
        out[0][()] = qlm
        out[1][()] = slm * shtns_cfg._l2
        out[2][()] = tlm * shtns_cfg._l2


def sht_analys_transposed(out, args, kwargs_dump):
    nargs = len(args)

    if nargs == 1:
        qlm = args[0].copy()
        qlm[shtns_cfg.lmax + 1 :] *= 0.5
        out[0][()] = shtns_cfg.synth(qlm) * shtns_cfg.wts
    elif nargs == 2:
        u, v = shtns_cfg.synth(args[0] * shtns_cfg._l_2, args[1] * shtns_cfg._l_2)
        out[0][()] = u * shtns_cfg.wts
        out[1][()] = v * shtns_cfg.wts
    elif nargs == 3:
        qlm = args[0].copy()
        qlm[shtns_cfg.lmax + 1 :] *= 0.5
        q, u, v = shtns_cfg.synth(
            qlm, args[1] * shtns_cfg._l_2, args[2] * shtns_cfg._l_2
        )
        out[0][()] = q * shtns_cfg.wts
        out[1][()] = u * shtns_cfg.wts
        out[2][()] = v * shtns_cfg.wts


def sht_synth_abstract_eval(*args, **kwargs):
    nargs = len(args)
    assert nargs <= 3
    for i in range(nargs):
        assert args[i].shape == (shtns_cfg.nlm,)
        assert args[i].dtype == jnp.complex128

    out_shape = (shtns_cfg.nlat, shtns_cfg.nphi)
    return ((out_shape, jnp.float64),) * nargs


def sht_analys_abstract_eval(*args, **kwargs):
    nargs = len(args)
    assert nargs <= 3
    for i in range(nargs):
        assert args[i].shape == (shtns_cfg.nlat, shtns_cfg.nphi)
        assert args[i].dtype == jnp.float64

    out_shape = (shtns_cfg.nlm,)
    return ((out_shape, jnp.complex128),) * nargs


sht_synth_jax = jaxbind.get_linear_call(
    sht_synth,
    sht_synth_transposed,
    sht_synth_abstract_eval,
    sht_analys_abstract_eval,
    func_can_batch=False,
)

sht_analys_jax = jaxbind.get_linear_call(
    sht_analys,
    sht_analys_transposed,
    sht_analys_abstract_eval,
    sht_synth_abstract_eval,
    func_can_batch=False,
)

eq

In [ ]:
theta_lm = np.zeros((len(eq.fd_grid), shtns_cfg.nlm), dtype=np.complex128)
u_tor_lm = np.zeros((len(eq.fd_grid), shtns_cfg.nlm), dtype=np.complex128)
u_pol_lm = np.zeros((len(eq.fd_grid), shtns_cfg.nlm), dtype=np.complex128)

lm44 = shtns_cfg.idx(4, 4)
z2 = (2 * eq.fd_grid - eq.r_grid[0] - eq.r_grid[-1]) ** 2
theta_lm[:, lm44] = 0.1 * (1 - 3 * z2 + 3 * z2**2 - z2**3)


def source(theta_lm, u_tor_lm, u_pol_lm):
    return 0


dt = 4e-4
step_fn = jax.jit(
    dynamical_solver(
        eq,
        imex.PC2(dt),
        source,
        sht_synth_jax,  # temporary
        sht_analys_jax,  # temporary
    )
)

In [ ]:
tke_t = []
buo_t = []

t = 0
iters = 1000
pbar = tqdm.tqdm(range(iters), bar_format="{l_bar}{bar:10}{r_bar}{bar:-10b}")
for i in pbar:
    t += dt

    c, theta_lm, u_tor_lm, u_pol_lm = step_fn(
        jnp.array(theta_lm),
        jnp.array(u_tor_lm),
        jnp.array(u_pol_lm),
    )

    if i % 25 == 0:
        eke_lm = kinetic_energy(eq, np.array(u_tor_lm), np.array(u_pol_lm))
        tke_t.append(
            np.sum(1.0 * eke_lm[:, shtns_cfg.m != 0])
            + np.sum(0.5 * eke_lm[:, shtns_cfg.m == 0])
        )
        buo_t.append(0.5 * integral(eq, np.array(theta_lm), shtns_cfg.m))

        pbar.set_postfix(eke_t=tke_t[-1], buo_t=buo_t[-1])

In [ ]:
plt.plot(buo_t)

In [ ]:
plt.plot(tke_t)

In [ ]:
import pyevtk
from pyevtk.hl import gridToVTK

cos_t = shtns_cfg.cos_theta
sin_t = np.sqrt(1 - cos_t**2)
r = eq.r_grid

n_r = len(r)
n_theta = shtns_cfg.nlat
n_phi = shtns_cfg.nphi + 1

print("n_r =", n_r, "- n_theta =", n_theta, "- n_phi =", n_phi)

f_spat = np.zeros((n_r, n_theta, n_phi - 1))
for ir in range(0, len(eq.r_grid)):
    f_spat[ir] = sht_synth_jax(theta_lm[ir])[0]

n_phi_full = shtns_cfg.mres * shtns_cfg.nphi + 1
phi = 2 * np.pi * np.arange(n_phi_full) / (n_phi_full - 1)
phi[-1] = phi[-2] + phi[1]

cos_t_l = r.reshape(n_r, 1) * cos_t.reshape(1, n_theta)
sin_t_l = r.reshape(n_r, 1) * sin_t.reshape(1, n_theta)
x = (sin_t_l.reshape(-1, 1) * np.cos(phi.reshape(1, n_phi_full))).reshape(
    n_r, n_theta, n_phi_full
)
y = (sin_t_l.reshape(-1, 1) * np.sin(phi.reshape(1, n_phi_full))).reshape(
    n_r, n_theta, n_phi_full
)
z = np.repeat(cos_t_l, n_phi_full).reshape(n_r, n_theta, n_phi_full)

f_pad = np.zeros((n_r, n_theta, n_phi_full))
for irep in range(shtns_cfg.mres):
    f_pad[:, :, irep * shtns_cfg.nphi : (irep + 1) * shtns_cfg.nphi] = f_spat
f_pad[:, :, -1] = f_spat[:, :, 0]

gridToVTK("conv-experiment", x, y, z, pointData={"temp": f_pad})